# 🎵 Suno Zara — free GPU music worker

This notebook runs the **ACE-Step 1.5** AI music model on Kaggle's free GPU. It picks up the songs people request in the
Suno Zara app, makes them **with real singing**, and saves them to your app.

**One-time setup (from the ⋮ / settings menu on the right, or "Settings" on mobile):**
1. **Accelerator → GPU T4 x2** (not P100)
2. **Internet → On** (needs a phone-verified Kaggle account)
3. **Add-ons → Secrets** → add these 3 secrets, then switch **each one on** for this notebook:
   - `DATABASE_URL`, `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY` (the same values you gave Render)
   - optional: `GROQ_API_KEY` (a free key from console.groq.com) for AI-written lyrics

**Then tap ▶▶ Run all.** The first song is ready about 10–15 minutes after starting (installing + downloading the model).
To keep it running with your phone locked, use **Save Version → Save & Run All**.

In [ ]:
# ⚙️ Settings: change if you like
RUN_HOURS = 3          # how long to keep making songs (Kaggle gives ~30 GPU hours per week)
SONG_SECONDS = 180     # length of new songs (3 minutes)
REPO = "https://github.com/cockroachbankersparty01-gif/suno-zara"

In [ ]:
# 1️⃣ Check GPU, internet and secrets
import os, subprocess, sys, time, json, urllib.request

def fail(msg):
    raise SystemExit("\n❌ " + msg + "\n")

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    fail("No GPU. Open Settings → Accelerator → choose 'GPU T4 x2', then Run all again.")
print("✅ GPU:", gpu.stdout.strip().splitlines()[0])
if "P100" in gpu.stdout:
    print("⚠️  P100 may be too old for the newest PyTorch. If you get CUDA errors, switch to GPU T4 x2.")

try:
    urllib.request.urlopen("https://github.com", timeout=15)
    print("✅ Internet is on")
except Exception:
    fail("No internet. Open Settings → Internet → On (verify your phone number on Kaggle if it's greyed out).")

secrets = {}
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for name in ["DATABASE_URL", "SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY", "GROQ_API_KEY"]:
        try:
            secrets[name] = client.get_secret(name).strip()
        except Exception:
            pass
except ImportError:
    pass
missing = [n for n in ["DATABASE_URL", "SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"] if not secrets.get(n)]
if missing:
    fail("Missing secrets: " + ", ".join(missing) + ". Open Add-ons → Secrets, add them, and switch each one ON for this notebook.")
print("✅ Secrets found:", ", ".join(k for k in secrets if secrets[k]))

In [ ]:
# 2️⃣ Install ACE-Step 1.5 (~3–5 min)
import os, subprocess
ACE = "/kaggle/working/ACE-Step-1.5"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Failed: {cmd}")
    return r.stdout
# ffmpeg first: ACE-Step needs it to save MP3s
if subprocess.run("which ffmpeg", shell=True).returncode != 0:
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
sh("pip install -q uv")
if not os.path.isdir(ACE):
    sh(f"git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git {ACE}")
print(sh(f"cd {ACE} && uv sync 2>&1 | tail -3"))
print("✅ ACE-Step installed")

In [ ]:
# 3️⃣ Start the ACE-Step music server on the GPU
import subprocess, time, urllib.request, os, json
ace_env = dict(os.environ,
    CUDA_VISIBLE_DEVICES="0",
    ACESTEP_DEVICE="cuda",
    ACESTEP_CONFIG_PATH="acestep-v15-turbo",
    ACESTEP_LM_BACKEND="pt",               # vLLM may not support the T4
    ACESTEP_INIT_LLM="false",              # the extra "thinking" LM isn't needed and saves memory
    ACESTEP_USE_FLASH_ATTENTION="false",   # T4 is too old for flash-attention
    ACESTEP_API_HOST="127.0.0.1",
    ACESTEP_API_PORT="8001",
)
ace_log = open("/kaggle/working/acestep.log", "w")
ace = subprocess.Popen("uv run acestep-api", shell=True, cwd=ACE, env=ace_env, stdout=ace_log, stderr=subprocess.STDOUT)

def health():
    try:
        return json.load(urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=5))
    except Exception:
        return None

t0 = time.time()
while not health():
    if ace.poll() is not None:
        print(open("/kaggle/working/acestep.log").read()[-4000:])
        raise SystemExit("❌ ACE-Step stopped while starting (log above).")
    if time.time() - t0 > 1200:
        print(open("/kaggle/working/acestep.log").read()[-4000:])
        raise SystemExit("❌ ACE-Step didn't start in 20 minutes (log above).")
    time.sleep(10)
print(f"✅ ACE-Step running ({int(time.time()-t0)}s)")

In [ ]:
# 4️⃣ Warm-up: make a 10-second test clip (downloads the model the first time, ~3–8 min)
import urllib.request, urllib.parse, json, time
def post(path, data, form=False):
    if form:
        body = urllib.parse.urlencode(data).encode()
        req = urllib.request.Request("http://127.0.0.1:8001" + path, data=body)
    else:
        req = urllib.request.Request("http://127.0.0.1:8001" + path, data=json.dumps(data).encode(), headers={"content-type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=60))

t0 = time.time()
task = post("/release_task", {"prompt": "happy acoustic guitar", "lyrics": "[Instrumental]", "audio_duration": "10",
                              "batch_size": "1", "audio_format": "mp3", "task_type": "text2music"}, form=True)["data"]["task_id"]
while True:
    item = post("/query_result", {"task_id_list": [task]})["data"][0]
    if item["status"] == 1:
        print(f"✅ Test clip made in {int(time.time()-t0)}s. The GPU is ready.")
        break
    if item["status"] == 2:
        print(open("/kaggle/working/acestep.log").read()[-4000:])
        raise SystemExit("❌ Test clip failed: " + str(item.get("result")))
    if time.time() - t0 > 1500:
        raise SystemExit("❌ Test clip took too long")
    time.sleep(5)

In [ ]:
# 5️⃣ Install the Suno Zara worker (~1–2 min)
import os, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Failed: {cmd}")
    return r.stdout
NODE = "/kaggle/working/node"
if not os.path.exists(f"{NODE}/bin/node"):
    sh("cd /kaggle/working && curl -fsSL https://nodejs.org/dist/v22.12.0/node-v22.12.0-linux-x64.tar.xz -o node.tar.xz "
       "&& mkdir -p node && tar -xJf node.tar.xz -C node --strip-components=1 && rm node.tar.xz")
os.environ["PATH"] = f"{NODE}/bin:" + os.environ["PATH"]
if subprocess.run("which ffmpeg", shell=True).returncode != 0:
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
APP = "/kaggle/working/suno-zara"
if os.path.isdir(APP):
    sh(f"cd {APP} && git pull -q")
else:
    sh(f"git clone --depth 1 {REPO} {APP}")
print(sh(f"cd {APP}/backend && npm install --no-audit --no-fund 2>&1 | tail -2 && npm run build 2>&1 | tail -2"))
print("✅ Worker ready")

In [ ]:
# 6️⃣ Make songs! Leave this running. New requests from the app are picked up automatically.
import subprocess, os, time, signal
w_env = dict(os.environ,
    DATABASE_URL=secrets["DATABASE_URL"],
    SUPABASE_URL=secrets["SUPABASE_URL"],
    SUPABASE_SERVICE_ROLE_KEY=secrets["SUPABASE_SERVICE_ROLE_KEY"],
    AUTH_MODE="supabase", STORAGE_DRIVER="supabase", STORAGE_BUCKET="media",
    MUSIC_PROVIDER="acestep", ACESTEP_URL="http://127.0.0.1:8001",
    IMAGE_PROVIDER="pollinations",
    DEFAULT_DURATION_SEC=str(SONG_SECONDS), MAX_DURATION_SEC="360",
    DB_POOL_MAX="3", WORKER_CONCURRENCY="1",
)
if secrets.get("GROQ_API_KEY"):
    w_env.update(LYRICS_PROVIDER="openai", LLM_BASE_URL="https://api.groq.com/openai/v1",
                 LLM_API_KEY=secrets["GROQ_API_KEY"], LLM_MODEL="llama-3.3-70b-versatile")
else:
    w_env.update(LYRICS_PROVIDER="mock")

worker = subprocess.Popen("node dist/worker.js", shell=True, cwd=f"{APP}/backend", env=w_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
print(f"🎵 Making songs for {RUN_HOURS} hours. Open the app and create a song!\n")
import threading
def times_up():
    print("⏰ Time's up: finishing the current song, then stopping.", flush=True)
    worker.send_signal(signal.SIGTERM)
timer = threading.Timer(RUN_HOURS * 3600, times_up)
timer.daemon = True
timer.start()
try:
    for line in worker.stdout:          # ends when the worker exits
        print(time.strftime("%H:%M"), line, end="", flush=True)
    worker.wait()
finally:
    timer.cancel()
    if worker.poll() is None:
        worker.terminate()
    ace.terminate()
print("👋 Worker stopped. Songs requested from now on wait until you run this notebook again.")
